# Skyline Online Courses: Confidence Intervals

Lesson 1.5 Practice Exercise. Computes confidence intervals for the mean of hours_studied and
proportion of Skyline enrollments where completion_status == "Completed" demonstrating the 95% interpretation.

Author: Kenya Harvey
Date: 08-11-2026

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px

In [2]:
skyline = pd.read_csv("../lesson-1-2-types-of-data/skyline_enrollments.csv")

n = len(skyline)
sample_mean = skyline["hours_studied"].mean()
sample_std = skyline["hours_studied"].std()

print(f"Sample size: n = {n}")
print(f"Sample mean: {sample_mean:.2f}")
print(f"Sample std dev: {sample_std:.2f}")

Sample size: n = 99
Sample mean: 29.37
Sample std dev: 2.81


In [3]:
# Standard error of the mean
standard_error = sample_std / np.sqrt(n)

# Critical value for 95% confidence
# Note: for small samples (n < 30), we'd use t-distribution.
# n=50 is a borderline case; both z and t are reasonable. We'll use z first.
z = 1.96

# Margin of error
margin_of_error = z * standard_error

# Confidence interval
ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print(f"Standard error: {standard_error:.2f}")
print(f"Margin of error (95%): {margin_of_error:.2f}")
print(f"95% CI for the mean: ({ci_lower:.2f}, {ci_upper:.2f})")

Standard error: 0.28
Margin of error (95%): 0.55
95% CI for the mean: (28.82, 29.92)


In [4]:
ci_scipy = stats.norm.interval(
    confidence=0.95,
    loc=sample_mean,
    scale=standard_error,
)

print(f"Manual 95% CI: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"scipy 95% CI:  ({ci_scipy[0]:.2f}, {ci_scipy[1]:.2f})")

Manual 95% CI: (28.82, 29.92)
scipy 95% CI:  (28.82, 29.92)


In [5]:
t_crit = stats.t.ppf(0.975, df=n-1)  # 0.975 because we want both tails to total 5%
margin_t = t_crit * standard_error
ci_t_lower = sample_mean - margin_t
ci_t_upper = sample_mean + margin_t

print(f"t-based 95% CI: ({ci_t_lower:.2f}, {ci_t_upper:.2f})")
print(f"t critical value at df={n-1}: {t_crit:.4f}")

t-based 95% CI: (28.81, 29.93)
t critical value at df=98: 1.9845


In [ ]:
np.random.seed(42)

true_mean = 250
true_std = 80
sample_size = 50
n_simulations = 1000

n_capturing_truth = 0
ci_widths = []

for _ in range(n_simulations):
    # Generate one new sample from the same population
    sample = np.random.normal(loc=true_mean, scale=true_std, size=sample_size)

    # Compute its sample mean and standard deviation
    s_mean = sample.mean()
    s_std = sample.std(ddof=1)

    # Compute the 95% CI
    se = s_std / np.sqrt(sample_size)
    moe = 1.96 * se
    lower = s_mean - moe
    upper = s_mean + moe

    ci_widths.append(upper - lower)

    # Check whether this CI contains the true mean
    if lower <= true_mean <= upper:
        n_capturing_truth += 1

capture_rate = n_capturing_truth / n_simulations
avg_width = np.mean(ci_widths)

print(f"Simulations run: {n_simulations}")
print(f"Confidence intervals that captured the true mean: {n_capturing_truth}")
print(f"Capture rate: {capture_rate:.1%}")
print(f"Average CI width: ${avg_width:.2f}")

Simulations run: 1000
Confidence intervals that captured the true mean: 945
Capture rate: 94.5%
Average CI width: $44.12


In [7]:
np.random.seed(42)

n_to_plot = 50  # plot the first 50 of the 1000 simulations
fig_data = []

for i in range(n_to_plot):
    sample = np.random.normal(loc=true_mean, scale=true_std, size=sample_size)
    s_mean = sample.mean()
    s_std = sample.std(ddof=1)
    se = s_std / np.sqrt(sample_size)
    moe = 1.96 * se
    lower = s_mean - moe
    upper = s_mean + moe

    captures = lower <= true_mean <= upper

    fig_data.append({
        "trial": i,
        "lower": lower,
        "upper": upper,
        "mean": s_mean,
        "captures": captures,
    })

fig_df = pd.DataFrame(fig_data)

fig = px.scatter(
    fig_df,
    x="mean",
    y="trial",
    color="captures",
    error_x_minus=fig_df["mean"] - fig_df["lower"],
    error_x=fig_df["upper"] - fig_df["mean"],
    title=f"50 simulated 95% CIs (true mean = {true_mean})",
    labels={"mean": "Sample mean", "trial": "Trial number", "captures": "Captures truth"},
)
fig.add_vline(x=true_mean, line_dash="dash", annotation_text="True mean")
fig.show()

If sampling annd computing were repeated many times, approximately 95% of the intervals would capture the true population average.


In [9]:
# Sample proportion
n = len(skyline)
n_completed = (skyline["completion_staus"] == "Completed").sum()
p_hat = n_completed / n

print(f"Sample size: {n}")
print(f"Completed_status: {n_completed}")
print(f"Sample proportion: {p_hat:.3f}")

Sample size: 99
Completed_status: 55
Sample proportion: 0.556


In [11]:
# Standard error of the proportion
se_prop = np.sqrt(p_hat * (1 - p_hat) / n)

# 95% CI
z = 1.96
moe_prop = z * se_prop
ci_prop_lower = p_hat - moe_prop
ci_prop_upper = p_hat + moe_prop

print(f"Standard error: {se_prop:.4f}")
print(f"Margin of error (95%): {moe_prop:.4f}")
print(f"95% CI for proportion Completed: ({ci_prop_lower:.3f}, {ci_prop_upper:.3f})")
print(f"Or: {p_hat:.1%} ± {moe_prop:.1%}")

Standard error: 0.0499
Margin of error (95%): 0.0979
95% CI for proportion Completed: (0.458, 0.653)
Or: 55.6% ± 9.8%


The Confidence interval is wide because sample size is n=99, which provides limitated precision. With n=99 and a proportion  of 55.6 percentage. The 95% confidence interval ranges from 45.8%-65.3%, while the sample proportion is 55.6%. Standard error decreases as sample size increases following the √n relationship. Formual setup: 0.02 = 1.96 × √(0.556 × 0.444 / n_new). To reduce the margin of error from approximately ±9.8 percentage points to ±2 percentage points, Skyline would need a substantially larger sample, roughly around 2,000 observations.
 
​	
 